In [1]:

import os
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gp
from skimage import io
from osgeo import gdal
import xarray as xr
import rioxarray as rxr
import earthaccess

# Login to earthaccess
earthaccess.login(persist=True)

# Times of which to find satellite images
temporal = ("2005-07-01T00:00:00", "2024-12-31T23:59:59")

# State bounding boxes
state_bboxes = {
    'Iowa':      (-96.6, 40.4, -90.1, 43.5),
    'Colorado':  (-109.1, 37.0, -102.0, 41.0),
    'Wisconsin': (-92.9, 42.5, -86.8, 47.1),
    'Missouri':  (-95.8, 36.0, -89.1, 40.6),
    'Nebraska':  (-104.1, 40.0, -95.3, 43.0),
}

states = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
target_bands = ["B04", "B05", "Fmask"]

for state in states:
    print(f"Searching data for {state}...")

    results = earthaccess.search_data(
        short_name=['HLSL30', 'HLSS30'],
        bounding_box=state_bboxes[state],
        temporal=temporal,
        count=100  # Increased from 3 to get meaningful coverage
    )

    print(f"  Found {len(results)} granules for {state}")

    # Collect only links matching target bands from ALL granules
    # Fix: iterate over granule objects and use .data_links(), not .endswith() on granules
    filtered_links = []
    for granule in results:
        links = granule.data_links()
        matching = [l for l in links if any(l.endswith(f"{b}.tif") for b in target_bands)]
        filtered_links.extend(matching)

    print(f"  Found {len(filtered_links)} matching band files for {state}")

    # Create output directory
    file_path = f"./cornYield/{state}/"
    os.makedirs(file_path, exist_ok=True)  # Fix: create directory if it doesn't exist

    # Download filtered links
    if filtered_links:
        earthaccess.download(filtered_links, local_path=file_path)
        print(f"  Downloaded files to {file_path}")
    else:
        print(f"  No matching files found for {state}, skipping download.")



Searching data for Iowa...
  Found 100 granules for Iowa
  Found 300 matching band files for Iowa


/opt/conda/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


QUEUEING TASKS | :   0%|          | 0/300 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/300 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/300 [00:00<?, ?it/s]

...

  Downloaded files to ./cornYield/Missouri/
Searching data for Nebraska...
  Found 100 granules for Nebraska
  Found 300 matching band files for Nebraska


QUEUEING TASKS | :   0%|          | 0/300 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/300 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/300 [00:00<?, ?it/s]

  Downloaded files to ./cornYield/Nebraska/


Please execute findCorn.py before continuing

In [2]:
files = os.listdir("./cornYield")
for f in sorted(files)[:20]:  # first 20 so it doesn't flood output
    print(f)

Colorado
Iowa
Missouri
Nebraska
Wisconsin


In [3]:

# folder = "./output/Iowa"
# files  = os.listdir(folder)

# granules = {}
# for f in files:
#     if not f.endswith(".tif"):
#         continue
#     for band in ["B04", "B05"]:
#         if band in f:
#             granule_id = f.split(f".{band}")[0]  # everything before the band name
#             if granule_id not in granules:
#                 granules[granule_id] = {}
#             granules[granule_id][band] = os.path.join(folder, f)

# print(f"Found {len(granules)} granule pairs\n")
# for gid, bands in list(granules.items())[:3]:   # just show first 3
#     print(f"Granule: {gid}")
#     for band, path in bands.items():
#         print(f"  {band} → {os.path.basename(path)}")
#     print()

In [4]:
# import rasterio
# import numpy as np

# # Grab the first granule
# first_granule = list(granules.values())[0]

# # Read B04 (red) and B05 (NIR)
# with rasterio.open(first_granule["B04"]) as src:
#     red = src.read(1).astype(float)

# with rasterio.open(first_granule["B05"]) as src:
#     nir = src.read(1).astype(float)

# print(f"Grid size: {red.shape}")           # how many pixels wide/tall
# print(f"Red  — min: {red.min():.0f}  max: {red.max():.0f}")
# print(f"NIR  — min: {nir.min():.0f}  max: {nir.max():.0f}")
# print(f"Unique values in red (first 10): {np.unique(red)[:10]}")

In [15]:

original_folder = "./savedfiles/Iowa"

granule_ids = []
for f in os.listdir("./cornYield/Iowa/"):
    if f.endswith(".B04.tif") and not f.startswith("corn_only"):
        granule_id = f.replace(".B04.tif", "")
        granule_ids.append(granule_id)

print(f"Found {len(granule_ids)} granules")
print("\nFirst 3:")
for g in granule_ids[:3]:
    print(" ", g)

Found 100 granules

First 3:
  HLS.L30.T15TUJ.2013106T170701.v2.0
  HLS.L30.T14TPP.2013106T170701.v2.0
  HLS.L30.T14TQP.2013106T170701.v2.0


In [16]:
granule_id = granule_ids[0]  # just the first one for now

b04_path   = os.path.join("./cornYield/Iowa/", f"{granule_id}.B04.tif")
b05_path   = os.path.join("./cornYield/Iowa/", f"{granule_id}.B05.tif")
fmask_path = os.path.join("./cornYield/Iowa/", f"{granule_id}.Fmask.tif")
corn_path  = os.path.join("./output/Iowa/", f"corn_only_{granule_id}.B04.tif")

print("B04:  ", b04_path)
print("B05:  ", b05_path)
print("Fmask:", fmask_path)
print("Corn: ", corn_path)

# Check they all actually exist on disk
for path in [b04_path, b05_path, fmask_path, corn_path]:
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {os.path.basename(path)}")

B04:   ./cornYield/Iowa/HLS.L30.T15TUJ.2013106T170701.v2.0.B04.tif
B05:   ./cornYield/Iowa/HLS.L30.T15TUJ.2013106T170701.v2.0.B05.tif
Fmask: ./cornYield/Iowa/HLS.L30.T15TUJ.2013106T170701.v2.0.Fmask.tif
Corn:  ./output/Iowa/corn_only_HLS.L30.T15TUJ.2013106T170701.v2.0.B04.tif
  ✓ HLS.L30.T15TUJ.2013106T170701.v2.0.B04.tif
  ✓ HLS.L30.T15TUJ.2013106T170701.v2.0.B05.tif
  ✓ HLS.L30.T15TUJ.2013106T170701.v2.0.Fmask.tif
  ✓ corn_only_HLS.L30.T15TUJ.2013106T170701.v2.0.B04.tif


In [17]:
with rasterio.open(b04_path)   as src: red   = src.read(1).astype(float)
with rasterio.open(b05_path)   as src: nir   = src.read(1).astype(float)
with rasterio.open(fmask_path) as src: fmask = src.read(1)
with rasterio.open(corn_path)  as src: corn  = src.read(1)

print(f"Red   — min: {red.min():.0f}   max: {red.max():.0f}")
print(f"NIR   — min: {nir.min():.0f}   max: {nir.max():.0f}")
print(f"Fmask — unique values: {np.unique(fmask)}")
print(f"Corn  — unique values: {np.unique(corn)}")

Red   — min: -9999   max: 14982
NIR   — min: -9999   max: 14821
Fmask — unique values: [ 64  66  68  70  72  74  76  78  80  84  96 100 128 130 132 134 136 138
 140 142 144 148 160 164 192 194 196 198 200 202 204 206 208 212 224 228
 240 244 255]
Corn  — unique values: [  0 255]


In [18]:
# Calculate NDVI for corn pixels

# Mask out invalid values (typically -9999 for HLS data)
valid_mask = (red > 0) & (nir > 0) & (corn == 255)

# Calculate NDVI: (NIR - Red) / (NIR + Red)
ndvi = np.full_like(red, np.nan)  # Initialize with NaN
denominator = nir + red

# Only calculate where denominator is not zero and pixels are valid
calc_mask = valid_mask & (denominator != 0)
ndvi[calc_mask] = (nir[calc_mask] - red[calc_mask]) / denominator[calc_mask]

print(f"NDVI — min: {np.nanmin(ndvi):.3f}   max: {np.nanmax(ndvi):.3f}")
print(f"Valid corn pixels: {np.sum(calc_mask):,}")
print(f"Mean NDVI for corn: {np.nanmean(ndvi):.3f}")# Calculate NDVI for corn pixels

# Mask out invalid values (typically -9999 for HLS data)
valid_mask = (red > 0) & (nir > 0) & (corn == 255)

# Calculate NDVI: (NIR - Red) / (NIR + Red)
ndvi = np.full_like(red, np.nan)  # Initialize with NaN
denominator = nir + red

# Only calculate where denominator is not zero and pixels are valid
calc_mask = valid_mask & (denominator != 0)
ndvi[calc_mask] = (nir[calc_mask] - red[calc_mask]) / denominator[calc_mask]

print(f"NDVI — min: {np.nanmin(ndvi):.3f}   max: {np.nanmax(ndvi):.3f}")
print(f"Valid corn pixels: {np.sum(calc_mask):,}")
print(f"Mean NDVI for corn: {np.nanmean(ndvi):.3f}")

NDVI — min: 0.889   max: 0.999
Valid corn pixels: 458
Mean NDVI for corn: 0.992
NDVI — min: 0.889   max: 0.999
Valid corn pixels: 458
Mean NDVI for corn: 0.992


In [19]:
# Process NDVI for all granules in Iowa

ndvi_results = []

for granule_id in granule_ids[:5]:  # Process first 5 for testing
    try:
        # File paths
        b04_path = f"./cornYield/Iowa/{granule_id}.B04.tif"
        b05_path = f"./cornYield/Iowa/{granule_id}.B05.tif"
        corn_path = f"./output/Iowa/corn_only_{granule_id}.B04.tif"
        
        # Check files exist
        if not all(os.path.exists(p) for p in [b04_path, b05_path, corn_path]):
            continue
            
        # Load data
        with rasterio.open(b04_path) as src: red = src.read(1).astype(float)
        with rasterio.open(b05_path) as src: nir = src.read(1).astype(float)
        with rasterio.open(corn_path) as src: corn = src.read(1)
        
        # Calculate NDVI for corn pixels
        valid_mask = (red > 0) & (nir > 0) & (corn == 255)
        ndvi = np.full_like(red, np.nan)
        denominator = nir + red
        calc_mask = valid_mask & (denominator != 0)
        ndvi[calc_mask] = (nir[calc_mask] - red[calc_mask]) / denominator[calc_mask]
        
        # Store results
        if np.sum(calc_mask) > 0:
            ndvi_results.append({
                'granule_id': granule_id,
                'mean_ndvi': np.nanmean(ndvi),
                'std_ndvi': np.nanstd(ndvi),
                'corn_pixels': np.sum(calc_mask)
            })
            
    except Exception as e:
        print(f"Error processing {granule_id}: {e}")

# Convert to DataFrame
ndvi_df = pd.DataFrame(ndvi_results)
print(f"Processed {len(ndvi_df)} granules")
print(ndvi_df.head())

Processed 5 granules
                           granule_id  mean_ndvi  std_ndvi  corn_pixels
0  HLS.L30.T15TUJ.2013106T170701.v2.0   0.991893  0.012430          458
1  HLS.L30.T14TPP.2013106T170701.v2.0   0.994569  0.003134           14
2  HLS.L30.T14TQP.2013106T170701.v2.0   0.992045  0.011784          109
3  HLS.L30.T14TPN.2013106T170724.v2.0   0.994232  0.005282            5
4  HLS.L30.T15TXJ.2013110T164240.v2.0   0.995594  0.000333            2


In [ ]:
import rasterio

# Process NDVI for all states

ndvi_results = []

for state in states:
    print(f"Processing NDVI for {state}...")
    
    # Get granule IDs for this state
    for f in os.listdir(f"./cornYield/{state}/"):
        if f.endswith(".B04.tif"):
            granule_id = f.replace(".B04.tif", "")
        try:
            b04_path = f"./cornYield/{state}/{granule_id}.B04.tif"
            b05_path = f"./cornYield/{state}/{granule_id}.B05.tif"
            
            # if not all(os.path.exists(p) for p in [b04_path, b05_path]):
            #     continue
                
            with rasterio.open(b04_path) as src: red = src.read(1).astype(float)
            with rasterio.open(b05_path) as src: nir = src.read(1).astype(float)
            
            # Calculate NDVI
            valid_mask = (red > 0) & (nir > 0)
            ndvi = np.full_like(red, np.nan)
            denominator = nir + red
            calc_mask = valid_mask & (denominator != 0)
            ndvi[calc_mask] = (nir[calc_mask] - red[calc_mask]) / denominator[calc_mask]
            
            if np.sum(calc_mask) > 0:
                ndvi_results.append({
                    'state': state,
                    'granule_id': granule_id,
                    'mean_ndvi': np.nanmean(ndvi),
                    'std_ndvi': np.nanstd(ndvi),
                    'valid_pixels': np.sum(calc_mask)
                })
                
        except Exception as e:
            print(f"Error processing {state}/{granule_id}: {e}")

ndvi_df = pd.DataFrame(ndvi_results)
print(f"\nProcessed {len(ndvi_df)} granules across all states")
print(ndvi_df.groupby('state').size())

Processing NDVI for Iowa...
Processing NDVI for Colorado...
Error processing Colorado/HLS.L30.T15TVH.2013131T170120.v2.0: ./cornYield/Colorado/HLS.L30.T15TVH.2013131T170120.v2.0.B04.tif: No such file or directory
Processing NDVI for Nebraska...
Error processing Nebraska/HLS.L30.T16SBH.2013119T163739.v2.0: ./cornYield/Nebraska/HLS.L30.T16SBH.2013119T163739.v2.0.B04.tif: No such file or directory
Error processing Nebraska/HLS.L30.T16SBH.2013119T163739.v2.0: ./cornYield/Nebraska/HLS.L30.T16SBH.2013119T163739.v2.0.B04.tif: No such file or directory

Processed 1495 granules across all states
state
Colorado     299
Iowa         300
Missouri     298
Nebraska     298
Wisconsin    300
dtype: int64
